# GRU — baseline (W=10, 5 semillas)

Modelo GRU de referencia sobre el dataset procesado (`data/processed/wednesday_low_and_slow_clean.csv`; ver `notebooks/Cleaning/limpieza_low_and_slow.ipynb`).

- **Arquitectura:** `GRU(32) → Dropout(0.3) → Dense(1, sigmoid)` (`src/models/gru/model.py`).
- **Datos:** secuencias W=10 con 59 features, variante `csv` (orden del archivo dentro de cada minuto), generadas por `scripts/gru/build_gru_sequences.py` (log con signo + StandardScaler ajustado solo con train).
- **Entrenamiento:** `class_weight` balanceado y EarlyStopping sobre `val_loss` con patience=10 (ADR 0013). Se entrena con 5 semillas (`fijar_semillas()` antes de construir cada modelo; ADR 0014) para reportar media y desviación estándar.
- **Solo se evalúa en validación.** Test no se carga en este notebook: se usa una única vez al final del proyecto.

## 1. Imports

In [1]:
import sys
sys.path.append("../..")  # raíz del repo, para poder importar src.*

import json
import os

import numpy as np

from src.models.gru.experiments import correr_experimento
from src.models.gru.train import compute_class_weights

## 2. Configuración y carga de train/val (W=10, variante csv)

In [2]:
CONFIG = {
    "units": 32,
    "dropout": 0.3,
    "learning_rate": 1e-3,
    "batch_size": 64,
    "epochs": 50,
    "patience": 10,
}
SEMILLAS = [0, 1, 2, 3, 4]

W = 10
DATA_DIR = "../../data/sequences"
MODELS_DIR = "../../models/gru/01_baseline"
RESULTS_DIR = "../../results/gru/01_baseline"

# Variante csv: archivos sin sufijo (ver scripts/gru/build_gru_sequences.py)
X_train = np.load(f"{DATA_DIR}/X_train_w{W}.npy")
y_train = np.load(f"{DATA_DIR}/y_train_w{W}.npy")
X_val = np.load(f"{DATA_DIR}/X_val_w{W}.npy")
y_val = np.load(f"{DATA_DIR}/y_val_w{W}.npy")

print(f"X_train: {X_train.shape} {X_train.dtype} | ataques: {int(y_train.sum())}")
print(f"X_val:   {X_val.shape} {X_val.dtype} | ataques: {int(y_val.sum())}")
assert X_train.shape[1:] == X_val.shape[1:] == (W, 59)

X_train: (310484, 10, 59) float32 | ataques: 8230
X_val:   (66496, 10, 59) float32 | ataques: 1884


## 3. Class weights

In [3]:
# Solo informativo: correr_experimento() calcula estos mismos pesos internamente
print("class_weight:", compute_class_weights(y_train))

class_weight: {0: 0.5136143773117974, 1: 18.862940461725394}


## 4. Entrenamiento con 5 semillas

`correr_experimento()` (`src/models/gru/experiments.py`) hace, por semilla: `clear_session` → `fijar_semillas` → construir → entrenar → guardar modelo, curvas y probabilidades de validación → métricas en validación. Al final guarda `baseline_w10_seeds.json` y devuelve la tabla por semilla con media y desviación estándar muestral (ddof=1).

In [4]:
tabla = correr_experimento(
    "baseline_w10", CONFIG, X_train, y_train, X_val, y_val, SEMILLAS, MODELS_DIR, RESULTS_DIR,
)


baseline_w10 — semilla 0

Epoch 1/50
4852/4852 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.9949 - auc: 0.9997 - loss: 0.0173 - precision: 0.8426 - recall: 0.9947 - val_accuracy: 0.9980 - val_auc: 0.9997 - val_loss: 0.0044 - val_precision: 0.9358 - val_recall: 0.9973
Epoch 2/50
4852/4852 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.9992 - auc: 0.9999 - loss: 0.0033 - precision: 0.9730 - recall: 0.9989 - val_accuracy: 0.9980 - val_auc: 0.9995 - val_loss: 0.0056 - val_precision: 0.9348 - val_recall: 0.9973
Epoch 3/50
4852/4852 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.9996 - auc: 0.9999 - loss: 0.0018 - precision: 0.9849 - recall: 0.9995 - val_accuracy: 0.9975 - val_auc: 0.9995 - val_loss: 0.0073 - val_precision: 0.9202 - val_recall: 0.9973
Epoch 4/50
4852/4852 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9998 - auc: 1.0000 - loss: 0.0012 - precision: 0.9928 - recall: 0.9995 - val_accuracy: 0.9998 - val_auc: 0.9995 - val_loss: 0.0012 - val_precision: 0.9947 - val_recall: 0

## 5. Resumen por semilla (validación)

In [5]:
tabla

,val_loss,mejor_epoca,epocas_entrenadas,tiempo_entrenamiento_s,roc_auc,pr_auc,recall_05,precision_05,umbral_f2,f2,recall_f2,precision_f2,fn_f2,fp_f2
0,0.000542,16.000000,26.000000,216.023646,0.999997,0.999894,0.997346,0.999468,0.07600,0.999044,0.998938,0.999469,2.000000,1.000000
1,0.000965,7.000000,17.000000,150.392958,0.999978,0.999429,0.995754,0.999467,0.23900,0.996494,0.995754,0.999467,8.000000,1.000000
2,0.002602,7.000000,17.000000,164.422105,0.999982,0.999489,0.994692,0.958078,0.83500,0.995006,0.994161,0.998401,11.000000,3.000000
3,0.000810,3.000000,13.000000,128.079830,0.999985,0.999652,0.994692,0.998934,0.14600,0.996708,0.996285,0.998404,7.000000,3.000000
4,0.002359,1.000000,11.000000,107.325499,0.999934,0.999207,0.996815,0.978125,0.74400,0.997027,0.996815,0.997875,6.000000,4.000000
media,0.001455,6.800000,16.800000,153.248808,0.999975,0.999534,0.995860,0.986814,0.40800,0.996856,0.996391,0.998723,6.800000,2.400000
std,0.000952,5.761944,5.761944,41.279857,0.000024,0.000257,0.001210,0.018496,0.35449,0.001449,0.001736,0.000713,3.271085,1.341641


## Conclusiones


- La GRU baseline (32 unidades, W=10, 59 features) alcanza en validación un F2 de 0.9969 ± 0.0014 y una PR-AUC de 0.9995 ± 0.0003 (5 semillas). En el umbral F2, el modelo comete 6.8 ± 3.3 falsos negativos y 2.4 ± 1.3 falsos positivos sobre 1,884 ataques.
- El desempeño es estable entre semillas en las métricas independientes del umbral (ROC-AUC y PR-AUC), pero no con el umbral fijo de 0.5: dos semillas bajan a una precision de 0.958 y 0.978. El umbral F2 elegido en validación corrige esa diferencia, por lo que la evaluación final usará el umbral F2 de cada semilla.
- El entrenamiento converge rápido (mejor época entre 1 y 16) y val_loss presenta picos entre épocas, mientras que el recall de validación se mantiene estable.
- Estas secuencias conservan el orden del CSV dentro de cada minuto. 02_ablaciones.ipynb evalúa cuánto depende el resultado de ese orden.